# Julián Serrano Chacón
# Mika Rodríguez Castro
  
# Práctica 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd
import time # para medir tiempos de ejecución
import soundfile as sf     
from ipywidgets import interact
from ipywidgets import fixed
import sys
import scipy.signal as sg
from tkinter import *

from consts import *
from tkinter import *

# graficos en el notebook
%matplotlib inline

SRATE = 48000 # Sample rate, para todo el notebook

CHUNK = 1024


class Osc:
    def __init__(self,freq=440.0,amp=1.0,phase=0.0,shape='sin', volume = 1):
        self.freq = freq
        self.amp = amp
        self.phase = phase
        self.frame = 0
        self.shape = shape
        self.volume = volume

    def next(self):    
        if self.shape=='sin':
            out = np.sin(2*np.pi*(np.arange(self.frame,self.frame+CHUNK))*self.freq/SRATE+self.phase)
        elif self.shape=='square':
            out = sg.square(2*np.pi*(np.arange(self.frame,self.frame+CHUNK))*self.freq/SRATE+self.phase)
        elif self.shape=='sawtooth':
            out = sg.sawtooth(2*np.pi*(np.arange(self.frame,self.frame+CHUNK))*self.freq/SRATE+self.phase)
        elif self.shape=='triangle':
            # Ojo, la triangular no existe como tal en scipy, pero podemos hacerla con dos sawtooth
            # el 2º parametro define la "rampa" la subida y bajada (ver documentacion)
            out = sg.sawtooth(2*np.pi*(np.arange(self.frame,self.frame+CHUNK))*self.freq/SRATE+self.phase,0.5)
        self.frame += CHUNK

        return np.float32(self.amp*out * self.volume)
    
    def getVol(self):
        return self.volume
    
    def getFreq(self):
        return self.freq
    
    def getAmp(self):
        return self.amp
    
    def setVol(self, volume):
        if volume > 1:
            self.volume = 1
        elif volume < 0:
            self.volume = 0
        else:
            self.volume = volume

    def setFreq(self, freq):
        self.freq = freq

    def setAmp(self, amp):
        if amp > 1:
            self.amp = 1
        elif amp < 0:
            self.amp = 0
        else:
            self.amp = amp


root=Tk()
def key_pressed(event):
    key = event.char
    if key in teclas:
        index = teclas.index(key) # sacamos posición en el string notas
        nota = notas[index]
        pitch = pitchs[index]
        print(f'tecla {key} nota {nota} pitch {pitch}')
    elif key == '-':
        print('note off')

    root.bind("<Key>",key_pressed)
    root.mainloop()


In [2]:
%%writefile consts.py

# mapeo de teclas del ordenador a notas en el piano
# utilizamos '.' para los sostenidos
teclas = "zsxdcvgbhnjmq2w3er5t6y7u"  # 2 de teclas filas 
notas =  "C.D.EF.G.A.Bc.d.ef.g.a.b"  # mapeadas a 2 octavas
#         octava baja||octava alta


# frecuencias de las notas asociadas a las teclas del teclado
# partimos del la=220Hz y generamos frecuencias de escala temperada
pitchs = [ 220*2.0**(i/12.0) for i in range(len(teclas))] 

# frecuencias asociadas a las notas midi de 0 a 127
# El LA central es la nota midi 70 y su frecuencia es 440
# construimos hacia abajo y hacia arriba el resto de notas
freqsMidi = [ 440*2.0**(i/12.0) for i in range(-69,59)]

Overwriting consts.py


#### Ejercicio 1 (Obligatorio)  
Utilizar el controlador de teclado propuesto para implementar un pequeño instrumento monofónico que utilize el sintetizador FM visto en clase como generador de señal.  
